# Phase 7 — Vòng Cải Thiện 1: Huấn Luyện Thử Nghiệm Từ Git (Nhánh `hoang`)

Notebook này phục vụ chạy quy trình thử nghiệm song song 2 nhánh (**Phase 7 Improvement Round 1**):
- `p0_legacy_full`: Huấn luyện với tiền xử lý truyền thống (stretch vuông 224x224 + ImageNet normalization).
- `p1_clip_full`: Huấn luyện với tiền xử lý chuẩn CLIP tương ứng deployment (resize bicubic cạnh ngắn 224 + center crop + CLIP normalization).

### Yêu cầu trước khi chạy:
1. **Môi trường Colab**: Chọn GPU CUDA (Runtime -> Change runtime type -> T4, L4, hoặc A100).
2. **Google Drive**: Tài khoản Drive của bạn phải có sẵn 3 tệp sau trong thư mục `MyDrive/Fatformer/`:
   - `MyDrive/Fatformer/phase7/train_crossdomain_v2.tar`
   - `MyDrive/Fatformer/pretrained/ViT-L-14.pt`
   - `MyDrive/Fatformer/pretrained/fatformer_4class_ckpt.pth`

### Quy trình chạy:
- Chạy tuần tự **Cell 1** -> **Cell 2** -> **Cell 3 (Smoke)** -> **Cell 4 (Train)**.
- Giữ nguyên cùng một phiên Colab giữa Cell 3 và Cell 4 để đồng bộ `PILOT_RUN_ID`.

In [1]:
# ==============================================================================
# CELL 1: KẾT NỐI GOOGLE DRIVE & KIỂM TRA GPU RUNTIME
# ==============================================================================
from google.colab import drive
import torch

print("[1/2] Đang kết nối Google Drive (vui lòng cấp quyền trong popup nếu được yêu cầu)...")
drive.mount('/content/drive')

print("\n[2/2] Kiểm tra GPU CUDA:")
assert torch.cuda.is_available(), "LỖI: Chưa chọn GPU! Vui lòng vào Runtime -> Change runtime type -> Chọn GPU trước khi chạy."
print(f"✓ Đã phát hiện GPU: {torch.cuda.get_device_name(0)}")
!nvidia-smi

[1/2] Đang kết nối Google Drive (vui lòng cấp quyền trong popup nếu được yêu cầu)...
Mounted at /content/drive

[2/2] Kiểm tra GPU CUDA:
✓ Đã phát hiện GPU: NVIDIA L4
Mon Nov  9 18:23:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   48C    P8             15W /   72W |       3MiB /  23034Mi

In [2]:
# ==============================================================================
# CELL 2: CLONE MÃ NGUỒN TỪ GIT (NHÁNH hoang) & KHÓA COMMIT 2665014
# ==============================================================================
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/fatformer-xla')
PINNED_COMMIT = '2665014'

if not REPO_DIR.exists():
    print("Đang clone repo từ GitHub (nhánh hoang)...")
    !git clone --branch hoang --single-branch https://github.com/LeeVHoangtk3/fatformer-xla.git /content/fatformer-xla
else:
    print(f"Repo đã tồn tại tại {REPO_DIR}. Đang fetch cập nhật...")
    !git -C /content/fatformer-xla fetch origin hoang

print(f"\nKhóa phiên bản tại commit: {PINNED_COMMIT}")
!git -C /content/fatformer-xla checkout {PINNED_COMMIT}

current_commit = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
print(f"\n✓ Commit hiện tại đang chạy: {current_commit}")
assert current_commit.startswith(PINNED_COMMIT), f"Lệch commit! Yêu cầu {PINNED_COMMIT}, thực tế {current_commit}"

Đang clone repo từ GitHub (nhánh hoang)...
Cloning into '/content/fatformer-xla'...
remote: Enumerating objects: 3343, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 3343 (delta 49), reused 73 (delta 47), pack-reused 3248 (from 2)
Receiving objects: 100% (3343/3343), 75.16 MiB | 20.31 MiB/s, done.
Resolving deltas: 100% (1906/1906), done.

Khóa phiên bản tại commit: 2665014
Note: switching to '2665014'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is 

In [3]:
# ==============================================================================
# CELL 3: GIAI ĐOẠN SMOKE TEST & XÁC THỰC TOÀN VẸN ĐẦU VÀO
# ==============================================================================
# Nhiệm vụ:
# 1. Kiểm tra hash SHA-256 của archive dữ liệu và pretrained weights trên Drive.
# 2. Giải nén dữ liệu sang SSD Colab (/content/phase7_pilot_data).
# 3. Chạy preflight + smoke 100 mini-batch cho cả 2 nhánh (p0_legacy_full & p1_clip_full).
# 4. Kiểm tra cập nhật optimizer và cơ chế chống lỗi gradient NaN (AMP update counter).
# ==============================================================================
from pathlib import Path
import subprocess

REPO = Path('/content/fatformer-xla')
PILOT_REPO = str(REPO)
PILOT_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True
).strip()

PILOT_STAGE = 'smoke'
PILOT_BATCH_SIZE = 16
PILOT_GRAD_ACCUM = 2

print(f"=== BẮT ĐẦU CHẠY GIAI ĐOẠN: {PILOT_STAGE.upper()} ===")
print(f"Commit: {PILOT_COMMIT}")
print(f"Cấu hình: Batch size = {PILOT_BATCH_SIZE}, Grad Accum = {PILOT_GRAD_ACCUM} (Effective Batch = {PILOT_BATCH_SIZE * PILOT_GRAD_ACCUM})")

exec(compile(
    (REPO / 'notebooks/task/phase7_improvement_round1/start_colab.py').read_text(),
    'start_colab.py', 'exec'
))

print("\n" + "="*80)
print("✓ GIAI ĐOẠN SMOKE TEST ĐÃ HOÀN TẤT THÀNH CÔNG!")
print("Vui lòng giữ nguyên phiên Colab hiện tại và chạy tiếp CELL 4 để bắt đầu huấn luyện.")
print("="*80)

=== BẮT ĐẦU CHẠY GIAI ĐOẠN: SMOKE ===
Commit: 26650142e76f790c0642c091a05dfc2dc3770cef
Cấu hình: Batch size = 16, Grad Accum = 2 (Effective Batch = 32)
Git branch: hoang; exact commit: 26650142e76f790c0642c091a05dfc2dc3770cef
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUN_ID: pilot_20261109_182359 STAGE: smoke GPU: NVIDIA L4
Git commit: 26650142e76f790c0642c091a05dfc2dc3770cef Drive results/weights: /content/drive/MyDrive/Fatformer/phase7/improvement_round1/pilot_20261109_182359
A smoke pass is an engineering check, not evidence of better quality or G7.2 PASS.
Saved: /content/drive/MyDrive/Fatformer/phase7/improvement_round1/pilot_20261109_182359/pilot_results_without_weights.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ GIAI ĐOẠN SMOKE TEST ĐÃ HOÀN TẤT THÀNH CÔNG!
Vui lòng giữ nguyên phiên Colab hiện tại và chạy tiếp CELL 4 để bắt đầu huấn luyện.


In [4]:
# ==============================================================================
# CELL 4: GIAI ĐOẠN HUẤN LUYỆN CHÍNH THỨC (5 EPOCHS CHO CẢ 2 NHÁNH)
# ==============================================================================
# Lưu ý: Chạy cell này trong CÙNG phiên Colab với Cell 3 để giữ nguyên PILOT_RUN_ID.
# Quá trình này sẽ:
# 1. Huấn luyện nhánh p0_legacy_full (tiền xử lý cũ).
# 2. Huấn luyện nhánh p1_clip_full (tiền xử lý CLIP chuẩn deployment).
# 3. Ghi vết từng lần cập nhật optimizer vào optimizer_updates.jsonl.
# 4. Đánh giá chọn checkpoint tốt nhất trên val_select.
# 5. Tự động sao lưu checkpoint, log và kết quả về Drive:
#    MyDrive/Fatformer/phase7/improvement_round1/<PILOT_RUN_ID>/
# ==============================================================================
from pathlib import Path

REPO = Path('/content/fatformer-xla')

PILOT_STAGE = 'train'

print(f"=== BẮT ĐẦU CHẠY GIAI ĐOẠN: {PILOT_STAGE.upper()} ===")
print(f"Kế thừa Run ID: {globals().get('PILOT_RUN_ID', 'Khởi tạo theo phiên')}")

exec(compile(
    (REPO / 'notebooks/task/phase7_improvement_round1/start_colab.py').read_text(),
    'start_colab.py', 'exec'
))

print("\n" + "="*80)
print("✓ HUẤN LUYỆN VÀ LỰA CHỌN MÔ HÌNH HOÀN TẤT THÀNH CÔNG!")
print("Toàn bộ checkpoint và kết quả đã được sao lưu an toàn về Google Drive.")
print("="*80)

=== BẮT ĐẦU CHẠY GIAI ĐOẠN: TRAIN ===
Kế thừa Run ID: pilot_20261109_182359
Git branch: hoang; exact commit: 26650142e76f790c0642c091a05dfc2dc3770cef
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUN_ID: pilot_20261109_182359 STAGE: train GPU: NVIDIA L4
Git commit: 26650142e76f790c0642c091a05dfc2dc3770cef Drive results/weights: /content/drive/MyDrive/Fatformer/phase7/improvement_round1/pilot_20261109_182359
A smoke pass is an engineering check, not evidence of better quality or G7.2 PASS.
Saved: /content/drive/MyDrive/Fatformer/phase7/improvement_round1/pilot_20261109_182359/pilot_results_without_weights.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ HUẤN LUYỆN VÀ LỰA CHỌN MÔ HÌNH HOÀN TẤT THÀNH CÔNG!
Toàn bộ checkpoint và kết quả đã được sao lưu an toàn về Google Drive.
